In [ ]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install pathlib
%pip install scikit-learn
%pip install imbalanced-learn
%pip install scipy
%pip install xgboost

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [1]:
import os
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(f"La máquina tiene {num_cores} núcleos disponibles.")
hilos_optimos = str(min(num_cores, 64)) 
print(f"Configurando OpenBLAS para usar {hilos_optimos} hilos máximos...")
os.environ['OPENBLAS_NUM_THREADS'] = hilos_optimos
os.environ['MKL_NUM_THREADS'] = hilos_optimos
os.environ['OMP_NUM_THREADS'] = hilos_optimos

La máquina tiene 224 núcleos disponibles.
Configurando OpenBLAS para usar 64 hilos máximos...


In [2]:
import numpy as np
import pandas as pd
import itertools
import json
from pathlib import Path
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, accuracy_score, recall_score,classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.combine import SMOTEENN


In [3]:
X = pd.read_pickle("Sets_Xy/X.pkl")
y = pd.read_pickle("Sets_Xy/y.pkl")

In [4]:
from sklearn.model_selection import train_test_split

#Division estratificada para muestras de cada clase a nivel de cada subset

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, 
    random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, 
    random_state=42)

In [5]:
#Encoding de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

mapeo_labels = pd.DataFrame({
    "label_original": le.classes_,
    "label_encoded": range(len(le.classes_))
})
print("Mapeo de etiquetas:\n", mapeo_labels)
class_names = le.classes_

Mapeo de etiquetas:
                label_original  label_encoded
0                      BENIGN              0
1                         Bot              1
2                        DDoS              2
3               DoS GoldenEye              3
4                    DoS Hulk              4
5            DoS Slowhttptest              5
6               DoS slowloris              6
7                 FTP-Patator              7
8                  Heartbleed              8
9                Infiltration              9
10                   PortScan             10
11                SSH-Patator             11
12    Web Attack  Brute Force             12
13  Web Attack  Sql Injection             13
14            Web Attack  XSS             14


In [6]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import ExtraTreesClassifier

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, method='tree', k=50, class_weight='balanced', random_state=42):
        self.method = method
        self.k = k
        self.class_weight = class_weight
        self.random_state = random_state
        self.selector = None
        self.feature_names_ = None
        self.support_mask_ = None

    def fit(self, X, y):
        self.feature_names_ = X.columns.tolist() if hasattr(X, 'columns') else list(range(X.shape[1]))
        
        X_array = X.values if hasattr(X, 'values') else X
        y_array = y.values if hasattr(y, 'values') else y
        
        total_features = X_array.shape[1]
        actual_k = min(self.k, total_features)
        
        if self.method == 'none':
            self.support_mask_ = np.ones(total_features, dtype=bool)
            return self

        if self.method == 'f_classif':
            self.selector = SelectKBest(score_func=f_classif, k=actual_k)
            self.selector.fit(X_array, y_array)
            self.support_mask_ = self.selector.get_support()

        elif self.method == 'tree':
            estimator = ExtraTreesClassifier(
                n_estimators=100,               
                max_depth=20,                   
                class_weight=self.class_weight,
                criterion='gini',               
                random_state=self.random_state, 
                n_jobs=-1
            )
            estimator.fit(X_array, y_array)
            importances = estimator.feature_importances_
            
            top_k_indices = np.argsort(importances)[-actual_k:]
            
            self.support_mask_ = np.zeros(total_features, dtype=bool)
            self.support_mask_[top_k_indices] = True

        return self

    def transform(self, X):
        if self.method == 'none':
            return X
            
        X_array = X.values if hasattr(X, 'values') else X
        X_transformed = X_array[:, self.support_mask_]

        if hasattr(X, 'columns'):
            selected_features = np.array(self.feature_names_)[self.support_mask_].tolist()
            return pd.DataFrame(X_transformed, columns=selected_features, index=X.index)
        else:
            return X_transformed

In [7]:
#Carga de datos post oversampling
import joblib

X_train_none = joblib.load("Sets_Oversampling/X_train_none.joblib")
y_train_none = joblib.load("Sets_Oversampling/y_train_none.joblib")

X_train_smote = joblib.load("Sets_Oversampling/X_train_smote.joblib")
y_train_smote = joblib.load("Sets_Oversampling/y_train_smote.joblib")

X_train_smote_enn = joblib.load("Sets_Oversampling/X_train_smote_enn.joblib")
y_train_smote_enn = joblib.load("Sets_Oversampling/y_train_smote_enn.joblib")

X_train_smote_tomek = joblib.load("Sets_Oversampling/X_train_smote_tomek.joblib")
y_train_smote_tomek = joblib.load("Sets_Oversampling/y_train_smote_tomek.joblib")

X_val = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian.pkl")
y_val = pd.DataFrame(y_val)

X_test = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian.pkl")
y_test = pd.DataFrame(y_test)

In [8]:
X_train_none_2 = joblib.load("Sets_Oversampling_2/X_train_none.joblib")
y_train_none_2 = joblib.load("Sets_Oversampling_2/y_train_none.joblib")

X_train_smote_2 = joblib.load("Sets_Oversampling_2/X_train_smote.joblib")
y_train_smote_2 = joblib.load("Sets_Oversampling_2/y_train_smote.joblib")

X_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/X_train_smote_enn.joblib")
y_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/y_train_smote_enn.joblib")

X_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/X_train_smote_tomek.joblib")
y_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/y_train_smote_tomek.joblib")


In [9]:
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

def balanced_class_weight(y):
    classes = np.unique(y)
    weights = compute_class_weight('balanced', classes=classes, y=y)
    return dict(zip(classes, weights))

def polynomial_class_weight(y, alpha=0.25):
    classes, counts = np.unique(y, return_counts=True)
    weights = 1.0 / np.power(counts, alpha)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def log_ratio_class_weight(y):
    classes, counts = np.unique(y, return_counts=True)
    total = np.sum(counts)
    weights = np.log(total / counts)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def effective_samples_class_weight(y, beta=0.999):
    classes, counts = np.unique(y, return_counts=True)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / effective_num
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def cost_sensitive_class_weight(y):
    classes, counts = np.unique(y, return_counts=True)
    max_count = np.max(counts)
    weights = max_count / counts
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def focal_class_weight_improved(y, gamma=2.0):

    classes, counts = np.unique(y, return_counts=True)
    probs = counts / np.sum(counts)
    weights = (1 - probs) ** gamma / probs
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

In [10]:
train_datasets = {
    'none': (X_train_none, y_train_none.values.ravel() if isinstance(y_train_none, pd.DataFrame) else y_train_none.ravel()),
    'smote_enn': (X_train_smote_enn, y_train_smote_enn.values.ravel() if isinstance(y_train_smote_enn, pd.DataFrame) else y_train_smote_enn.ravel()),
    'smote_tomek': (X_train_smote_tomek, y_train_smote_tomek.values.ravel() if isinstance(y_train_smote_tomek, pd.DataFrame) else y_train_smote_tomek.ravel())
}

y_val_1d = y_val.values.ravel() if isinstance(y_val, pd.DataFrame) else y_val.ravel()
y_test_1d = y_test.values.ravel() if isinstance(y_test, pd.DataFrame) else y_test.ravel()

In [11]:
train_datasets_2 = {
    'none': (X_train_none_2, y_train_none_2.values.ravel() if isinstance(y_train_none_2, pd.DataFrame) else y_train_none_2.ravel()),
    'smote': (X_train_smote_2, y_train_smote_2.values.ravel() if isinstance(y_train_smote_2, pd.DataFrame) else y_train_smote_2.ravel()),
    'smote_enn': (X_train_smote_enn_2, y_train_smote_enn_2.values.ravel() if isinstance(y_train_smote_enn_2, pd.DataFrame) else y_train_smote_enn_2.ravel()),
    'smote_tomek': (X_train_smote_tomek_2, y_train_smote_tomek_2.values.ravel() if isinstance(y_train_smote_tomek_2, pd.DataFrame) else y_train_smote_tomek_2.ravel())
}

In [12]:
import joblib
X_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_none.joblib")
y_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_none.joblib")

X_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote.joblib")
y_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote.joblib")

X_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_enn.joblib")
y_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_enn.joblib")

X_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_tomek.joblib")
y_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_tomek.joblib")

X_val_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian_grouped.pkl")
X_test_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian_grouped.pkl")

y_val_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_val_grouped.pkl")
y_test_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_test_grouped.pkl")

In [13]:
train_datasets_grouped = {
    'none': (X_train_none_grouped, y_train_none_grouped.values.ravel() if isinstance(y_train_none_grouped, pd.DataFrame) else y_train_none_grouped.ravel()),
    'smote': (X_train_smote_grouped, y_train_smote_grouped.values.ravel() if isinstance(y_train_smote_grouped, pd.DataFrame) else y_train_smote_grouped.ravel()),
    'smote_enn': (X_train_smote_enn_grouped, y_train_smote_enn_grouped.values.ravel() if isinstance(y_train_smote_enn_grouped, pd.DataFrame) else y_train_smote_enn_grouped.ravel()),
    'smote_tomek': (X_train_smote_tomek_grouped, y_train_smote_tomek_grouped.values.ravel() if isinstance(y_train_smote_tomek_grouped, pd.DataFrame) else y_train_smote_tomek_grouped.ravel())
}

In [15]:
def save_confusion_matrix_xgb(log_folder, y_true, y_pred, trial_number, dataset_name, phase="val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'confusion matrix - {dataset_name} trial {trial_number} ({phase})')
    plt.ylabel('true label')
    plt.xlabel('predicted label')
    plt.savefig(f'{log_folder}/{dataset_name}_trial_{trial_number}_xgb_{phase}_cm.png', bbox_inches='tight')
    plt.close()

In [ ]:
import os
import optuna
import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight

log_folder = "Logs_XGBoost_Baseline_1"
os.makedirs(log_folder, exist_ok=True)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

datasets_to_test = ['none', 'smote', 'smote_enn', 'smote_tomek']
resultados_globales = {}

for current_dataset in datasets_to_test:
    print(f"\niniciando estudio enfocado en set agrupado: {current_dataset}")
    
    def objective_xgb(trial):
        x_train_raw, y_train_raw = train_datasets_grouped[current_dataset]
        total_features = x_train_raw.shape[1]

        weight_func_name = trial.suggest_categorical('weight_func', ['none', 'balanced', 'log_ratio', 'cost_sensitive'])
        
        if weight_func_name == 'none':
            weight_dict_fs = None
            sample_weights_xgb = None
            
        elif weight_func_name == 'balanced':
            weight_dict_fs = 'balanced'
            sample_weights_xgb = compute_sample_weight('balanced', y=y_train_raw)
            
        else:
            if weight_func_name == 'log_ratio':
                weight_dict_fs = log_ratio_class_weight(y_train_raw)
            elif weight_func_name == 'cost_sensitive':
                weight_dict_fs = cost_sensitive_class_weight(y_train_raw)

            web_boost = trial.suggest_float('web_boost', 1.0, 3.0) 
            bot_boost = trial.suggest_float('bot_boost', 1.5, 3.5) 
            
            if 12 in weight_dict_fs: weight_dict_fs[12] *= web_boost
            if 1 in weight_dict_fs: weight_dict_fs[1] *= bot_boost
                
            map_func = np.vectorize(lambda x: weight_dict_fs.get(x, 1.0))
            sample_weights_xgb = map_func(y_train_raw)

        fs_method = trial.suggest_categorical('fs_method', ['none', 'tree'])
        
        if fs_method == 'none':
            k_features = total_features
        else:
            k_features = trial.suggest_int('k_features', 20, total_features)
        
        selector = FeatureSelector(method=fs_method, k=k_features, class_weight=weight_dict_fs) 
        x_train_fs = selector.fit_transform(x_train_raw, y_train_raw)
        x_val_fs = selector.transform(X_val_imp_grouped)
        
        xgb_params = {
            'n_estimators': trial.suggest_int('n_estimators', 150, 400),
            'max_depth': trial.suggest_int('max_depth', 6, 15), 
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'tree_method': 'hist', 
            'device': 'cuda',
            'objective': 'multi:softmax',
            'num_class': 13,
            'random_state': 42
        }
        
        model = xgb.XGBClassifier(**xgb_params)
        
        model.fit(x_train_fs, y_train_raw, sample_weight=sample_weights_xgb)
        
        y_pred = model.predict(x_val_fs)
        
        f1_mac = f1_score(y_val_grouped, y_pred, average='macro')
        report = classification_report(y_val_grouped, y_pred, zero_division=0)
        
        save_confusion_matrix_xgb(log_folder,y_val_grouped, y_pred, trial.number, current_dataset, phase="val")
        
        log_msg = f"""dataset: {current_dataset} | trial: {trial.number}
fs method: {fs_method} | k features: {k_features}
params xgb: {trial.params}

f1 macro: {f1_mac:.4f}
{report}"""
        
        with open(f"{log_folder}/{current_dataset}_trial_{trial.number}.log", 'w') as f:
            f.write(log_msg)
            
        return f1_mac

    study_name = f"xgb_opt_grouped_{current_dataset}"
    study = optuna.create_study(direction='maximize', study_name=study_name)
    study.optimize(objective_xgb, n_trials=100, n_jobs=1)
    
    resultados_globales[current_dataset] = {
        'best_trial': study.best_trial.number,
        'best_f1_macro': study.best_value,
        'best_params': study.best_params
    }
    
    best_log_msg = f"""mejor trial para dataset agrupado (xgboost): {current_dataset}
trial numero: {study.best_trial.number}
f1 macro alcanzado: {study.best_value:.4f}
hiperparametros:
{study.best_params}
"""
    with open(f"{log_folder}/mejor_trial_{current_dataset}.log", 'w') as f:
        f.write(best_log_msg)

    print(f"\nresultados finales para {current_dataset}")
    print(f"mejor trial: {study.best_trial.number}")
    print(f"mejor f1 macro: {study.best_value:.4f}")
    print(f"mejores params: {study.best_params}\n")

informe_final = "informe final: competencia de datasets agrupados (xgboost)\n"

for dataset, metricas in resultados_globales.items():
    informe_final += f"dataset: {dataset}\n"
    informe_final += f"- mejor trial: {metricas['best_trial']}\n"
    informe_final += f"- f1 macro alcanzado: {metricas['best_f1_macro']:.4f}\n"
    informe_final += f"- hiperparametros ganadores:\n"
    for param, value in metricas['best_params'].items():
        informe_final += f"* {param}: {value}\n"

ganador_absoluto = max(resultados_globales, key=lambda x: resultados_globales[x]['best_f1_macro'])
informe_final += f"\nganador absoluto: {ganador_absoluto} (f1 macro: {resultados_globales[ganador_absoluto]['best_f1_macro']:.4f})\n"

print(informe_final)

with open(f"{log_folder}/resumen_global_resultados.txt", 'w') as f:
    f.write(informe_final)

print(f"el informe final ha sido guardado en {log_folder}/resumen_global_resultados.txt")

[I 2026-03-09 16:41:18,646] A new study created in memory with name: xgb_opt_grouped_none



iniciando estudio enfocado en set agrupado: none


[I 2026-03-09 16:41:37,193] Trial 0 finished with value: 0.9378755991469474 and parameters: {'weight_func': 'log_ratio', 'web_boost': 2.2048068932838767, 'bot_boost': 1.9922156872911738, 'fs_method': 'tree', 'k_features': 51, 'n_estimators': 175, 'max_depth': 14, 'learning_rate': 0.019181078383558047, 'subsample': 0.9939469660964133, 'colsample_bytree': 0.8254151137360504}. Best is trial 0 with value: 0.9378755991469474.
[I 2026-03-09 16:42:00,533] Trial 1 finished with value: 0.9188458470924818 and parameters: {'weight_func': 'balanced', 'fs_method': 'tree', 'k_features': 54, 'n_estimators': 261, 'max_depth': 10, 'learning_rate': 0.015058824819604178, 'subsample': 0.7969410182403996, 'colsample_bytree': 0.9124281379258941}. Best is trial 0 with value: 0.9378755991469474.
[I 2026-03-09 16:42:14,471] Trial 2 finished with value: 0.7244570952150051 and parameters: {'weight_func': 'cost_sensitive', 'web_boost': 2.866350479583524, 'bot_boost': 2.991438473290504, 'fs_method': 'tree', 'k_fea


resultados finales para none
mejor trial: 65
mejor f1 macro: 0.9617
mejores params: {'weight_func': 'none', 'fs_method': 'tree', 'k_features': 59, 'n_estimators': 387, 'max_depth': 9, 'learning_rate': 0.1505830237684683, 'subsample': 0.9988985881667181, 'colsample_bytree': 0.6463159010113986}


iniciando estudio enfocado en set agrupado: smote


[I 2026-03-09 17:21:03,143] Trial 0 finished with value: 0.9521438562098451 and parameters: {'weight_func': 'none', 'fs_method': 'none', 'n_estimators': 378, 'max_depth': 12, 'learning_rate': 0.03680873188474869, 'subsample': 0.6068822422914893, 'colsample_bytree': 0.7015304781124833}. Best is trial 0 with value: 0.9521438562098451.
[I 2026-03-09 17:21:28,979] Trial 1 finished with value: 0.9236192168209821 and parameters: {'weight_func': 'log_ratio', 'web_boost': 2.217782206229149, 'bot_boost': 2.7494684625220938, 'fs_method': 'tree', 'k_features': 49, 'n_estimators': 251, 'max_depth': 10, 'learning_rate': 0.02633660773116107, 'subsample': 0.7596925216186603, 'colsample_bytree': 0.7596728228490404}. Best is trial 0 with value: 0.9521438562098451.
[I 2026-03-09 17:21:50,629] Trial 2 finished with value: 0.9140050207001362 and parameters: {'weight_func': 'balanced', 'fs_method': 'none', 'n_estimators': 246, 'max_depth': 14, 'learning_rate': 0.031207335493366985, 'subsample': 0.716765236


resultados finales para smote
mejor trial: 65
mejor f1 macro: 0.9529
mejores params: {'weight_func': 'none', 'fs_method': 'none', 'n_estimators': 377, 'max_depth': 7, 'learning_rate': 0.0922100112806016, 'subsample': 0.6213367240678677, 'colsample_bytree': 0.859904818858891}


iniciando estudio enfocado en set agrupado: smote_enn


[I 2026-03-09 17:59:52,592] Trial 0 finished with value: 0.9257425856842686 and parameters: {'weight_func': 'balanced', 'fs_method': 'tree', 'k_features': 57, 'n_estimators': 187, 'max_depth': 13, 'learning_rate': 0.011396412280312155, 'subsample': 0.9990272608593455, 'colsample_bytree': 0.6226153419240232}. Best is trial 0 with value: 0.9257425856842686.
[I 2026-03-09 18:00:30,694] Trial 1 finished with value: 0.948333934152289 and parameters: {'weight_func': 'balanced', 'fs_method': 'tree', 'k_features': 66, 'n_estimators': 393, 'max_depth': 15, 'learning_rate': 0.06751687718843075, 'subsample': 0.7822692727939909, 'colsample_bytree': 0.8373667752129365}. Best is trial 1 with value: 0.948333934152289.
[I 2026-03-09 18:00:52,467] Trial 2 finished with value: 0.9166630333960085 and parameters: {'weight_func': 'log_ratio', 'web_boost': 2.268544433903277, 'bot_boost': 2.0185951315698194, 'fs_method': 'tree', 'k_features': 53, 'n_estimators': 230, 'max_depth': 8, 'learning_rate': 0.021150


resultados finales para smote_enn
mejor trial: 70
mejor f1 macro: 0.9509
mejores params: {'weight_func': 'balanced', 'fs_method': 'none', 'n_estimators': 353, 'max_depth': 15, 'learning_rate': 0.18643013865156902, 'subsample': 0.8758674229116665, 'colsample_bytree': 0.9826219307845969}


iniciando estudio enfocado en set agrupado: smote_tomek


[I 2026-03-09 18:40:40,796] Trial 0 finished with value: 0.8472441847818721 and parameters: {'weight_func': 'cost_sensitive', 'web_boost': 1.0978147519927222, 'bot_boost': 2.3416958876022447, 'fs_method': 'none', 'n_estimators': 302, 'max_depth': 11, 'learning_rate': 0.1459791292052643, 'subsample': 0.9285148422289317, 'colsample_bytree': 0.738603454733671}. Best is trial 0 with value: 0.8472441847818721.
[I 2026-03-09 18:41:10,559] Trial 1 finished with value: 0.9435539329801561 and parameters: {'weight_func': 'none', 'fs_method': 'tree', 'k_features': 57, 'n_estimators': 264, 'max_depth': 13, 'learning_rate': 0.012901591243160574, 'subsample': 0.8014428324392674, 'colsample_bytree': 0.6800790161751695}. Best is trial 1 with value: 0.9435539329801561.
[I 2026-03-09 18:41:32,824] Trial 2 finished with value: 0.7923553104622066 and parameters: {'weight_func': 'cost_sensitive', 'web_boost': 1.41864327391792, 'bot_boost': 2.103759223032418, 'fs_method': 'tree', 'k_features': 36, 'n_estima


resultados finales para smote_tomek
mejor trial: 38
mejor f1 macro: 0.9544
mejores params: {'weight_func': 'none', 'fs_method': 'none', 'n_estimators': 161, 'max_depth': 8, 'learning_rate': 0.16975347514334424, 'subsample': 0.9337508865655523, 'colsample_bytree': 0.7759752037198928}

informe final: competencia de datasets agrupados (xgboost)
dataset: none
- mejor trial: 65
- f1 macro alcanzado: 0.9617
- hiperparametros ganadores:
* weight_func: none
* fs_method: tree
* k_features: 59
* n_estimators: 387
* max_depth: 9
* learning_rate: 0.1505830237684683
* subsample: 0.9988985881667181
* colsample_bytree: 0.6463159010113986
dataset: smote
- mejor trial: 65
- f1 macro alcanzado: 0.9529
- hiperparametros ganadores:
* weight_func: none
* fs_method: none
* n_estimators: 377
* max_depth: 7
* learning_rate: 0.0922100112806016
* subsample: 0.6213367240678677
* colsample_bytree: 0.859904818858891
dataset: smote_enn
- mejor trial: 70
- f1 macro alcanzado: 0.9509
- hiperparametros ganadores:
* w

In [ ]:
import os
import xgboost as xgb
import numpy as np
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

print("Iniciando evaluación final en Test Set para los campeones de XGBoost...\n")

log_folder = "Logs_XGBoost_Baseline_1"

best_xgb_configs = {
    'none': {
        'trial': 65, 
        'fs_method': 'tree', 
        'k_features': 59, 
        'weight_func': 'none',
        'xgb_params': {
            'n_estimators': 387, 'max_depth': 9, 'learning_rate': 0.1505830237684683, 
            'subsample': 0.9988985881667181, 'colsample_bytree': 0.6463159010113986, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softmax', 
            'num_class': 13, 'random_state': 42
        }
    },
    'smote': {
        'trial': 65, 
        'fs_method': 'none', 
        'k_features': 80,
        'weight_func': 'none',
        'xgb_params': {
            'n_estimators': 377, 'max_depth': 7, 'learning_rate': 0.0922100112806016, 
            'subsample': 0.6213367240678677, 'colsample_bytree': 0.859904818858891, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softmax', 
            'num_class': 13, 'random_state': 42
        }
    },
    'smote_enn': {
        'trial': 70, 
        'fs_method': 'none', 
        'k_features': 80, 
        'weight_func': 'balanced',
        'xgb_params': {
            'n_estimators': 353, 'max_depth': 15, 'learning_rate': 0.18643013865156902, 
            'subsample': 0.8758674229116665, 'colsample_bytree': 0.9826219307845969, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softmax', 
            'num_class': 13, 'random_state': 42
        }
    },
    'smote_tomek': {
        'trial': 38, 
        'fs_method': 'none', 
        'k_features': 80, 
        'weight_func': 'none',
        'xgb_params': {
            'n_estimators': 161, 'max_depth': 8, 'learning_rate': 0.16975347514334424, 
            'subsample': 0.9337508865655523, 'colsample_bytree': 0.7759752037198928, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softmax', 
            'num_class': 13, 'random_state': 42
        }
    }
}

for current_dataset, config in best_xgb_configs.items():
    print(f"Evaluando modelo XGBoost campeón para: {current_dataset.upper()}")
    
    x_train_best, y_train_best = train_datasets_grouped[current_dataset]
    
    weight_dict_fs = None
    sample_weights_xgb = None
    
    if config['weight_func'] == 'balanced':
        weight_dict_fs = 'balanced'
        sample_weights_xgb = compute_sample_weight('balanced', y=y_train_best)
        print("Aplicando pesos 'balanced'...")

    print(f"Aplicando Feature Selection: {config['fs_method']} (k={config['k_features']})...")
    final_selector = FeatureSelector(method=config['fs_method'], k=config['k_features'], class_weight=weight_dict_fs)
    x_train_final = final_selector.fit_transform(x_train_best, y_train_best)
    x_test_final = final_selector.transform(X_test_imp_grouped) 
    
    print("Entrenando XGBoost en la GPU..")
    final_xgb = xgb.XGBClassifier(**config['xgb_params'])
    
    if sample_weights_xgb is not None:
        final_xgb.fit(x_train_final, y_train_best, sample_weight=sample_weights_xgb)
    else:
        final_xgb.fit(x_train_final, y_train_best)
    
    print("Evaluando en el set de test puro...")
    y_pred_test = final_xgb.predict(x_test_final)
    
    test_f1_mac = f1_score(y_test_grouped, y_pred_test, average='macro')
    test_report = classification_report(y_test_grouped, y_pred_test, zero_division=0)
    
    print(f"Resultados finales - F1 macro: {test_f1_mac:.4f}\n")
    
    save_confusion_matrix_xgb(log_folder,
        y_true=y_test_grouped, 
        y_pred=y_pred_test, 
        trial_number=config['trial'], 
        dataset_name=current_dataset, 
        phase="test_final"
    )
    
    log_msg = f"""Evaluación final con Test
Modelo: XGBoost
Dataset: {current_dataset} | Feature Selection: {config['fs_method']} (k={config['k_features']})
Weight Function: {config['weight_func']}
Params XGB: {config['xgb_params']}

F1 macro (Test): {test_f1_mac:.4f}

Métricas por clase (Test):
{test_report}
"""
    log_filename = f"{log_folder}/xgb_final_test_metrics_{current_dataset}.log"
    with open(log_filename, 'w', encoding='utf-8') as f:
        f.write(log_msg)

print("Pipeline de pruebas completado con éxito para XGBoost.")
print(f"Revisa la carpeta {log_folder}/ para ver las matrices y los reportes.")

Iniciando evaluación final en Test Set para los campeones de XGBoost...

Evaluando modelo XGBoost campeón para: NONE
Aplicando Feature Selection: tree (k=59)...
Entrenando XGBoost en la GPU..
Evaluando en el set de test puro...
Resultados finales - F1 macro: 0.9038

Evaluando modelo XGBoost campeón para: SMOTE
Aplicando Feature Selection: none (k=80)...
Entrenando XGBoost en la GPU..
Evaluando en el set de test puro...
Resultados finales - F1 macro: 0.9204

Evaluando modelo XGBoost campeón para: SMOTE_ENN
Aplicando pesos 'balanced'...
Aplicando Feature Selection: none (k=80)...
Entrenando XGBoost en la GPU..
Evaluando en el set de test puro...
Resultados finales - F1 macro: 0.9123

Evaluando modelo XGBoost campeón para: SMOTE_TOMEK
Aplicando Feature Selection: none (k=80)...
Entrenando XGBoost en la GPU..
Evaluando en el set de test puro...
Resultados finales - F1 macro: 0.9212

Pipeline de pruebas completado con éxito para XGBoost.
Revisa la carpeta Logs_XGBoost_Baseline_1/ para ver l

In [16]:
#Revisando los resultados hasta el momento he identificado que el problema para mejorar aun mas la clasificacion se encuentra en la clase 1 (Ataque Bot) y las clases 8 y 9 (Heartbleed y Infiltration)

#Para esto voy a hacer tests cambiando el enfoque de XGBoost al emplear multi:softprob en vez de multi:softmax para obtener probabilidades de clase con base en las cuales se pueda configurar el modelo. Caso clase 1

#Tambien siguiendo la recomendacion de mi tutor y de lo definido en mi propuesta voy a aplicar Focal Loss para penalizar fallos en clases minoritarias y mejorar la especializacion sobre las mismas. Caso clases 8 y 9

#Ademas el bot boost que emplee para la clase 1 voy a restringirlo con base en mis resultados acotando su intervalo para Optuna

In [ ]:
import os
import optuna
import numpy as np
import xgboost as xgb
from scipy.optimize import minimize
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_sample_weight
import warnings
import gc

warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

log_folder = "Logs_XGBoost_Baseline_2"
os.makedirs(log_folder, exist_ok=True)

datasets_to_test = ['none', 'smote', 'smote_enn', 'smote_tomek']
resultados_globales = {}

for current_dataset in datasets_to_test:
    print(f"\nIniciando estudio enfocado en set agrupado con XGBoost: {current_dataset}")
    
    def objective_xgb(trial):
        x_train_raw, y_train_raw = train_datasets_grouped[current_dataset]
        total_features = x_train_raw.shape[1]

        # Agrego focal a las opciones de pesos
        weight_func_name = trial.suggest_categorical('weight_func', ['none', 'balanced', 'focal'])
        
        if weight_func_name == 'none':
            weight_dict_fs = None
            sample_weights_xgb = None
            
        elif weight_func_name == 'balanced':
            weight_dict_fs = 'balanced'
            sample_weights_xgb = compute_sample_weight('balanced', y=y_train_raw)
            
        else:
            if weight_func_name == 'focal':
                gamma = trial.suggest_float('focal_gamma', 1.0, 5.0)
                weight_dict_fs = focal_class_weight_improved(y_train_raw, gamma=gamma)

            # Ajusto el intervalo de BotBoost con base en resultados
            web_boost = trial.suggest_float('web_boost', 1.0, 2.0) 
            bot_boost = trial.suggest_float('bot_boost', 1.0, 1.5) 
            
            if 12 in weight_dict_fs: weight_dict_fs[12] *= web_boost
            if 1 in weight_dict_fs: weight_dict_fs[1] *= bot_boost
                
            map_func = np.vectorize(lambda x: weight_dict_fs.get(x, 1.0))
            sample_weights_xgb = map_func(y_train_raw)

        fs_method = trial.suggest_categorical('fs_method', ['none', 'tree'])
        
        if fs_method == 'none':
            k_features = total_features
        else:
            k_features = trial.suggest_int('k_features', 20, total_features)
        
        selector = FeatureSelector(method=fs_method, k=k_features, class_weight=weight_dict_fs) 
        x_train_fs = selector.fit_transform(x_train_raw, y_train_raw)
        x_val_fs = selector.transform(X_val_imp_grouped)
        
        xgb_params = {
            'n_estimators': trial.suggest_int('n_estimators', 150, 400),
            'max_depth': trial.suggest_int('max_depth', 6, 15), 
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'tree_method': 'hist', 
            'device': 'cpu',
            # Cambio a probabilidades para el objetivo de XGBoost
            'objective': 'multi:softprob',
            'num_class': 13,
            'random_state': 42,
            'max_bin': 64
        }
        
        x_train_fs = x_train_fs.astype(np.float32)
        x_val_fs = x_val_fs.astype(np.float32)
        
        model = xgb.XGBClassifier(**xgb_params)
        model.fit(x_train_fs, y_train_raw, sample_weight=sample_weights_xgb)
        
        # Extraigo probabilidades en lugar de la clase final
        y_pred_proba = model.predict_proba(x_val_fs)
        
        # Optimización de Umbrales con Pesos de Decisión
        def threshold_loss(weights):
            weighted_probs = y_pred_proba * weights
            y_pred_tuned = np.argmax(weighted_probs, axis=1)
            return -f1_score(y_val_grouped, y_pred_tuned, average='macro')
        
        initial_weights = np.ones(13)
        bounds = [(0.1, 5.0) for _ in range(13)]
        
        res = minimize(threshold_loss, initial_weights, method='Nelder-Mead', bounds=bounds)
        best_threshold_weights = res.x
        
        final_probs = y_pred_proba * best_threshold_weights
        y_pred = np.argmax(final_probs, axis=1)
        
        f1_mac = f1_score(y_val_grouped, y_pred, average='macro')
        report = classification_report(y_val_grouped, y_pred, zero_division=0)
        
        trial.set_user_attr("best_threshold_weights", best_threshold_weights.tolist())
        
        save_confusion_matrix_xgb(log_folder, y_val_grouped, y_pred, trial.number, current_dataset, phase="val")
        
        log_msg = f"""dataset: {current_dataset} | trial: {trial.number}
fs method: {fs_method} | k features: {k_features}
params xgb: {trial.params}
best threshold weights: {np.round(best_threshold_weights, 3).tolist()}

f1 macro: {f1_mac:.4f}
{report}"""
        
        with open(f"{log_folder}/{current_dataset}_trial_{trial.number}.log", 'w') as f:
            f.write(log_msg)

        del model
        del selector
        del x_train_fs
        del x_val_fs
        del y_pred_proba
        del final_probs
        gc.collect()
            
        return f1_mac

    study_name = f"xgb_optimizado_{current_dataset}"
    study = optuna.create_study(direction='maximize', study_name=study_name)
    study.optimize(objective_xgb, n_trials=75, n_jobs=1, gc_after_trial=True) #Paso de 100 a 75 para ver como van los resultados dando igual trials suficientes para notar patrones de convergencia
    
    resultados_globales[current_dataset] = {
        'best_trial': study.best_trial.number,
        'best_f1_macro': study.best_value,
        'best_params': study.best_params,
        'best_weights': study.best_trial.user_attrs["best_threshold_weights"]
    }
    
    best_log_msg = f"""Mejor trial para dataset agrupado (xgboost optimizado): {current_dataset}
Trial numero: {study.best_trial.number}
F1 macro alcanzado: {study.best_value:.4f}
Hiperparametros:
{study.best_params}
best threshold weights:
{np.round(study.best_trial.user_attrs["best_threshold_weights"], 3).tolist()}
"""
    with open(f"{log_folder}/mejor_trial_{current_dataset}.log", 'w') as f:
        f.write(best_log_msg)

    print(f"\nResultados finales para {current_dataset}")
    print(f"Mejor trial: {study.best_trial.number}")
    print(f"Mejor F1 Macro: {study.best_value:.4f}\n")

[I 2026-03-15 17:06:09,039] A new study created in memory with name: xgb_optimizado_none



Iniciando estudio enfocado en set agrupado con XGBoost: none
